<a href="https://colab.research.google.com/github/mlbyhoang/customer-segmentation/blob/main/notebooks/03_kmeans_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 – K-Means Clustering

## Objective

Identify customer segments based on demographic,
contract, and product characteristics using K-Means.

The workflow includes feature preprocessing,
dimensionality reduction, cluster evaluation,
and customer segment profiling.

The original dataset is proprietary and is not
included in this repository.

In [ ]:
# Import Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
from PIL import Image as PILImage
from IPython.display import display
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import Image

warnings.filterwarnings('ignore')

In [ ]:
# Load prepared dataset
DATA_PATH = Path("../data/customers_prepared.csv")

df = pd.read_csv(DATA_PATH)


In [ ]:
# Feature Selection

numerical_features = [
    "customer_age",
    "contract_duration",
    "monthly_fee",
    "upfront_payment"
]

categorical_features = [
    "sim_only",
    "provider_category",
    "device_category"
]

features = numerical_features + categorical_features

X = df[features].copy()

In [ ]:
# Data Preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

X_transformed = preprocessor.fit_transform(X)


In [ ]:
# Principal Component Analysis


# Reduce dimensions for visualization
pca = PCA(n_components=3)

X_pca = pca.fit_transform(X_transformed)

# Explained variance
explained_variance = pca.explained_variance_ratio_

print(
    "Explained variance:",
    explained_variance.round(3)
)

print(
    "Total explained variance:",
    explained_variance.sum().round(3)
)


In [ ]:
# Optimal Number of Clusters


silhouette_scores = []

for n_clusters in range(2, 11):

    kmeans = KMeans(
        n_clusters=n_clusters,
        init="k-means++",
        n_init=20,
        max_iter=500,
        random_state=42
    )

    labels = kmeans.fit_predict(X_transformed)

    score = silhouette_score(
        X_transformed,
        labels,
        sample_size=min(5000, len(X_transformed)),
        random_state=42
    )

    silhouette_scores.append(score)

# Visualize silhouette scores
plt.figure(figsize=(9, 5))

plt.plot(
    range(2, 11),
    silhouette_scores,
    marker="o"
)

plt.title("Silhouette Score by Number of Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.xticks(range(2, 11))
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# K-Means Clustering


# Select number of clusters with highest silhouette score
optimal_n_clusters = (
    np.argmax(silhouette_scores) + 2
)

print(f"Optimal number of clusters: {optimal_n_clusters}")

# Train final K-Means model
kmeans = KMeans(
    n_clusters=optimal_n_clusters,
    init="k-means++",
    n_init=20,
    max_iter=500,
    random_state=42
)

df["cluster"] = kmeans.fit_predict(X_transformed)

In [ ]:
# Create PCA visualization dataframe
df_pca = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2", "PC3"]
)

df_pca["cluster"] = df["cluster"].to_numpy()

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

scatter = ax.scatter(
    df_pca["PC1"],
    df_pca["PC2"],
    df_pca["PC3"],
    c=df_pca["cluster"],
    cmap="tab10",
    alpha=0.6,
    s=12
)

ax.set(
    xlabel="PC1",
    ylabel="PC2",
    zlabel="PC3",
    title="Customer Segments – PCA Visualization"
)

fig.colorbar(scatter, ax=ax, label="Cluster")

plt.tight_layout()
plt.show()


In [ ]:
# Cluster Profiling

# Numerical cluster profiles
numerical_profile = (
    df.groupby("cluster")[
        [
            "customer_age",
            "contract_duration",
            "monthly_fee",
            "upfront_payment",
            "profitability"
        ]
    ]
    .median()
    .round(2)
)

# Most frequent categorical values per cluster
categorical_profile = (
    df.groupby("cluster")[
        [
            "provider_category",
            "device_category",
            "sim_only"
        ]
    ]
    .agg(
        lambda x: x.mode().iloc[0]
        if not x.mode().empty else np.nan
    )
)

# Cluster sizes
cluster_sizes = df.groupby("cluster").size().rename(
    "customer_count"
)

# Combine profiles
cluster_profile = pd.concat(
    [
        cluster_sizes,
        numerical_profile,
        categorical_profile
    ],
    axis=1
)

display(cluster_profile)


## Conclusion

This notebook presents a K-Means clustering workflow
for customer segmentation.

The analysis includes:

- Numerical and categorical feature preprocessing
- PCA-based dimensionality reduction for visualization
- Cluster evaluation using Silhouette Scores
- K-Means model training
- Customer segment profiling

The workflow illustrates how unsupervised learning
can support data-driven customer segmentation.

Actual customer data and business results are excluded
for confidentiality reasons.